# 00 — Setup lab

Confirms your environment works and introduces the three clients every later lab uses.

**Prerequisites**
1. `pwsh scripts/01_connect_azure.ps1` has written a `.env`
2. A Foundry project exists (portal or `02_provision_core.ps1`)
3. `gpt-4o-mini` and `text-embedding-3-small` are deployed

Run the cells in order.

## 1. Load configuration

`scripts/ai103.py` reads `.env` and builds authenticated clients. Every lab starts
with these three lines.

In [ ]:
import sys, pathlib

sys.path.insert(0, str(pathlib.Path.cwd().parents[0] / "scripts"))
from ai103 import cfg, credential, chat_client, project_client, ask, show_usage, portal_link

print("subscription :", cfg["AZURE_SUBSCRIPTION_ID"])
print("resource grp :", cfg["AZURE_RESOURCE_GROUP"])
print("region       :", cfg["AZURE_LOCATION"])
print("project      :", cfg["AZURE_AI_PROJECT_ENDPOINT"])
print("inference    :", cfg["AZURE_OPENAI_ENDPOINT"])

Note the two endpoints. They are **different** and the exam tests the distinction:

| Endpoint | Shape | Used by | For |
|---|---|---|---|
| **Project** | `.../api/projects/<name>` | `AIProjectClient` | Agents, connections, evaluations, indexes, traces |
| **Inference** | `https://<resource>.services.ai.azure.com` | `AzureOpenAI`, `ChatCompletionsClient` | Chat, embeddings, images |

Rule of thumb: if it is *about* a model call, use the inference endpoint. If it is
about anything the project owns, use the project endpoint.

## 2. Authenticate without keys

`DefaultAzureCredential` walks a chain of credential sources — environment
variables, workload identity, managed identity, Azure CLI, Azure PowerShell, VS
Code, interactive browser — and uses the first that yields a token. Locally that is
your `az login`. In production it is a managed identity, with **no code change**.

That property is exactly why "keyless credentials" is a study-guide bullet.

In [ ]:
import datetime as dt

token = credential().get_token("https://cognitiveservices.azure.com/.default")
expires = dt.datetime.fromtimestamp(token.expires_on, dt.timezone.utc)

print(f"token acquired, expires {expires:%Y-%m-%d %H:%M UTC}")
print("scope        : https://cognitiveservices.azure.com/.default")
print("\nData-plane scopes you will meet in this course:")
for name, scope in {
    "Foundry / OpenAI": "https://cognitiveservices.azure.com/.default",
    "Azure AI Search": "https://search.azure.com/.default",
    "Storage": "https://storage.azure.com/.default",
    "ARM (management)": "https://management.azure.com/.default",
}.items():
    print(f"  {name:<18} {scope}")

## 3. Inspect the project

`AIProjectClient` is the control plane. Two things matter most: **deployments**
(which models you can call) and **connections** (which external resources the
project can reach — Search, Storage, other AI services).

In [ ]:
project = project_client()

print("Deployments")
print("-" * 60)
for d in project.deployments.list():
    kind = getattr(d, "type", "")
    model = getattr(d, "model_name", "") or getattr(d, "name", "")
    version = getattr(d, "model_version", "")
    print(f"  {d.name:<28} {model} {version} {kind}")

print("\nConnections")
print("-" * 60)
conns = list(project.connections.list())
if not conns:
    print("  none yet — unit 05.1 adds an Azure AI Search connection")
for c in conns:
    print(f"  {c.name:<28} {c.type}")

> **Exam note.** A *connection* stores the endpoint plus auth for an external
> resource so agents and tools can use it without you passing credentials around.
> Connections can be scoped to a single project or shared across the resource.

## 4. Call a model

You call the **deployment name**, not the model name. They happen to match here
because that is the sane convention, but they are independent — you could deploy
`gpt-4o` under the name `production-chat` and call that.

In [ ]:
client = chat_client()

response = client.chat.completions.create(
    model=cfg["MODEL_MINI"],  # <- deployment name
    messages=[
        {"role": "system", "content": "You are terse. Answer in one sentence."},
        {"role": "user", "content": "What is the difference between a Foundry resource and a Foundry project?"},
    ],
    temperature=0.2,
)

print(response.choices[0].message.content)
print()
show_usage(response)

### Token accounting

Get in the habit of reading `usage` now. Cost, rate limits, and latency all track
tokens, and unit 02.5 builds real observability on top of these numbers.

- `prompt_tokens` — everything you sent, including the system message and any
  retrieved context. RAG makes this the dominant cost.
- `completion_tokens` — what the model produced. Billed at a higher rate.
- Reasoning models (`o4-mini`) also bill hidden `reasoning_tokens` inside
  `completion_tokens_details`.

In [ ]:
# Rough cost of the call above. Rates change — check the pricing page for real numbers.
RATE_IN, RATE_OUT = 0.15 / 1_000_000, 0.60 / 1_000_000  # gpt-4o-mini, USD per token

u = response.usage
cost = u.prompt_tokens * RATE_IN + u.completion_tokens * RATE_OUT
print(f"approx cost: ${cost:.8f}")
print(f"you could make ~{int(1 / cost):,} calls like this for $1")

## 5. Embeddings

Same client, different endpoint. Vectors are the basis of the retrieval units.

In [ ]:
from ai103 import embed

vectors = embed([
    "Azure AI Search supports vector, keyword, and hybrid retrieval.",
    "The cat sat on the mat.",
])

print(f"dimensions: {len(vectors[0])}")


def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    na = sum(x * x for x in a) ** 0.5
    nb = sum(y * y for y in b) ** 0.5
    return dot / (na * nb)


query = embed("How do I do hybrid search?")[0]
for text, vec in zip(["search sentence", "cat sentence"], vectors):
    print(f"  similarity to {text:<16}: {cosine(query, vec):.4f}")

The relevant sentence scores far higher. That is the whole idea behind vector
search, and unit 05.1 makes Azure AI Search do it at scale.

## 6. Where things live in the portal

In [ ]:
for label in ["project", "resource_group", "cost", "quotas"]:
    print(f"{label:<16} {portal_link(label)}")

## Exercise

Three small tasks. Solutions at the bottom of [quiz.md](quiz.md).

1. Call `MODEL_MINI` twice with the same prompt — once at `temperature=0.0` and
   once at `temperature=1.5`. Run each three times. Describe what changed.
2. Deliberately request a deployment name that does not exist. Read the error.
   Which HTTP status comes back, and how does it differ from an auth failure?
3. Use `project.deployments.get(<name>)` to print the SKU and capacity of your
   chat deployment. Compare it with the value shown in the portal.

In [ ]:
# Your work here.

## Mark this unit complete

Ask the agent: *"mark 00_setup complete"* — or edit `../../ai103-learning.json` by
hand.

**Next:** [01.1 — Choose services and models](../01_plan_and_manage/01_choose_services_and_models/README.md)